In [1]:
!pip install -q accelerate datasets

In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secret = UserSecretsClient().get_secret('HF_TOKEN')
login(token=secret)
print('Logged in to HuggingFace.')

Logged in to HuggingFace.


In [3]:
import torch
import numpy as np
import json
import warnings
from typing import Dict, List, Tuple
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers

warnings.filterwarnings('ignore')

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'transformers : {transformers.__version__}')
print(f'torch        : {torch.__version__}')
print(f'Device       : {DEVICE}')
if 'cuda' in DEVICE:
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    print(f'VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

transformers : 4.57.1
torch        : 2.8.0+cu126
Device       : cuda:0
GPU          : Tesla T4
VRAM         : 15.6 GB


In [4]:
MODEL_NAME          = 'meta-llama/Llama-3.2-1B'

# Which MLP layer to edit (0-indexed). 16 layers total in LLaMA-1B.
# We find the best one automatically below; this is the fallback default.
EDIT_LAYER          = 8

# v* optimisation (paper eq. 4)
V_LR                = 0.001       # learning rate
V_STEPS             = 100        # gradient steps
V_KL_WEIGHT         = 0.0925   # KL penalty weight (low = stronger edit)
V_GRAD_CLIP         = 4.0       # gradient norm clip

# Rank-one update (paper eq. 7)
# MOM2_LAMBDA         = 15000.0   # λ — regularisation for C^{-1}

print('Config ready.')
print(f'  Edit layer  : {EDIT_LAYER}')
print(f'  v* lr/steps : {V_LR} / {V_STEPS}')
print(f'  KL weight   : {V_KL_WEIGHT}')
# print(f'  λ (mom2)    : {MOM2_LAMBDA}')

Config ready.
  Edit layer  : 8
  v* lr/steps : 0.001 / 100
  KL weight   : 0.0925


In [5]:
print(f'Loading {MODEL_NAME} …')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype      = torch.float32,   # float32 required for stable gradient optimisation
    device_map = {'': DEVICE},    # pin ALL layers to one GPU — avoids cross-device errors
)
model.eval()

NUM_LAYERS = model.config.num_hidden_layers
HIDDEN     = model.config.hidden_size
INTER      = model.config.intermediate_size

print(f'Layers       : {NUM_LAYERS}')
print(f'Hidden dim   : {HIDDEN}')
print(f'Intermediate : {INTER}')
print(f'Parameters   : {sum(p.numel() for p in model.parameters())/1e6:.0f} M')
print(f'Weight device: {next(model.parameters()).device}')

# Quick sanity check
ids = tokenizer('The capital of France is', return_tensors='pt').input_ids.to(DEVICE)
with torch.no_grad():
    out = model.generate(ids, max_new_tokens=5, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
print('Sanity check :', tokenizer.decode(out[0], skip_special_tokens=True))

Loading meta-llama/Llama-3.2-1B …


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

2026-03-22 20:45:30.130917: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774212330.324587      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774212330.378754      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774212330.864756      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774212330.864784      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774212330.864787      55 computation_placer.cc:177] computation placer alr

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Layers       : 16
Hidden dim   : 2048
Intermediate : 8192
Parameters   : 1236 M
Weight device: cuda:0
Sanity check : The capital of France is Paris. It is the


In [6]:
dataset = load_dataset('azhx/counterfact', split='train')
print(f'CounterFact samples: {len(dataset)}')
# Show one entry
s = dataset[0]
r = s['requested_rewrite']
print(f"\ncase_id  : {s['case_id']}")
print(f"subject  : {r['subject']}")
print(f"prompt   : {r['prompt']}")
print(f"true     : {r['target_true']['str']}")
print(f"new      : {r['target_new']['str']}")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-05d11247db7abc(…):   0%|          | 0.00/11.1M [00:00<?, ?B/s]

data/test-00000-of-00001-bacb83500fca49a(…):   0%|          | 0.00/1.25M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19728 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2191 [00:00<?, ? examples/s]

CounterFact samples: 19728

case_id  : 0
subject  : Danielle Darrieux
prompt   : The mother tongue of {} is
true     : French
new      : English


In [7]:
# ── Module shortcuts ─────────────────────────────────────────────────────────

def get_mlp(layer: int):
    return model.model.layers[layer].mlp

def get_down_proj(layer: int) -> torch.nn.Parameter:
    """down_proj weight — shape (HIDDEN=2048, INTER=8192)."""
    return model.model.layers[layer].mlp.down_proj.weight


# ── Token helpers ─────────────────────────────────────────────────────────────

def find_subject_range(prompt: str, subject: str) -> Tuple[int, int]:
    """Return (start, end) token positions of subject inside prompt."""
    p_ids = tokenizer.encode(prompt)
    for candidate in [' ' + subject, subject]:
        s_ids = tokenizer.encode(candidate, add_special_tokens=False)
        # drop leading-space token if tokenizer prepended one
        sp_id = tokenizer.encode(' ', add_special_tokens=False)
        if sp_id and s_ids[:len(sp_id)] == sp_id:
            s_ids = s_ids[len(sp_id):]
        for i in range(len(p_ids) - len(s_ids) + 1):
            if p_ids[i : i+len(s_ids)] == s_ids:
                return i, i + len(s_ids)
    # fallback: treat last token as the anchor
    return len(p_ids)-1, len(p_ids)


def get_target_tok(target: str) -> int:
    """First token id of target (prepended with space)."""
    return tokenizer.encode(' ' + target, add_special_tokens=False)[0]


# ── Inference ────────────────────────────────────────────────────────────────

@torch.no_grad()
def prob(prompt: str, target: str) -> float:
    ids    = tokenizer(prompt, return_tensors='pt').input_ids.to(DEVICE)
    logits = model(ids).logits[0, -1].float()
    return torch.softmax(logits, dim=-1)[get_target_tok(target)].item()


@torch.no_grad()
def generate(prompt: str, n: int = 20) -> str:
    ids  = tokenizer(prompt, return_tensors='pt').input_ids.to(DEVICE)
    out  = model.generate(
        ids,
        attention_mask = torch.ones_like(ids),
        max_new_tokens = n,
        do_sample      = False,
        pad_token_id   = tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)


print('Helpers ready.')

Helpers ready.


In [8]:
def causal_trace(prompt: str, subject: str, target_true: str) -> Dict[int, float]:
    """
    Noise the subject token embeddings, then patch each MLP output back
    to its clean value one layer at a time.  The layer whose patch gives
    the biggest P(target_true) recovery is where the fact is stored.
    (Simplified version of Fig 3 in the paper.)
    """
    ids = tokenizer(prompt, return_tensors='pt').input_ids.to(DEVICE)
    _, end   = find_subject_range(prompt, subject)
    subj_pos = end - 1
    t_id     = get_target_tok(target_true)

    # ── 1. Clean run — cache all MLP outputs ────────────────────────────────
    clean_mlp = {}
    hooks = []
    for l in range(NUM_LAYERS):
        def make_save(li):
            def h(mod, inp, out): clean_mlp[li] = out.detach().clone()
            return h
        hooks.append(get_mlp(l).register_forward_hook(make_save(l)))
    with torch.no_grad():
        clean_logits = model(ids).logits[0, -1].float()
    for h in hooks: h.remove()
    clean_p = torch.softmax(clean_logits, dim=-1)[t_id].item()

    # ── 2. Corrupted run — add Gaussian noise to subject token embeddings ────
    embed = model.model.embed_tokens(ids).detach().clone()
    noise = torch.randn_like(embed) * embed.std() * 3   # large noise
    embed_noisy = embed.clone()
    embed_noisy[0, :end] = embed_noisy[0, :end] + noise[0, :end]

    with torch.no_grad():
        corrupt_logits = model(
            inputs_embeds=embed_noisy
        ).logits[0, -1].float()
    corrupt_p = torch.softmax(corrupt_logits, dim=-1)[t_id].item()

    print(f'Clean P({target_true}) = {clean_p:.4f}')
    print(f'Corrupt P({target_true}) = {corrupt_p:.4f}')
    print(f'Signal gap = {clean_p - corrupt_p:.4f}')
    print()

    # ── 3. For each MLP layer: patch its clean output at subj_pos ────────────
    scores = {}
    print('Layer | P(target) after patch | Recovery')
    print('-' * 45)
    for l in range(NUM_LAYERS):
        def make_patch(li, pos):
            def h(mod, inp, out):
                p = out.clone()
                p[0, pos] = clean_mlp[li][0, pos]
                return p
            return h

        hook = get_mlp(l).register_forward_hook(make_patch(l, subj_pos))
        with torch.no_grad():
            patched_logits = model(
                inputs_embeds=embed_noisy
            ).logits[0, -1].float()
        hook.remove()

        p_val    = torch.softmax(patched_logits, dim=-1)[t_id].item()
        recovery = (p_val - corrupt_p) / (clean_p - corrupt_p + 1e-8)
        scores[l] = recovery
        marker = ' ◄── best candidate' if recovery == max(scores.values()) else ''
        print(f'  {l:2d}  | {p_val:.4f}                  | {recovery:.3f}{marker}')

    best = max(scores, key=scores.get)
    print(f'\n→ Best layer by causal trace: {best}')
    return scores, best


s0 = dataset[0]['requested_rewrite']
trace_scores, trace_best = causal_trace(
    prompt      = s0['prompt'].format(s0['subject']),
    subject     = s0['subject'],
    target_true = s0['target_true']['str'],
)

# Override EDIT_LAYER only if causal trace finds something in the middle layers
# (layers 5-12). Layer 0 is never correct — it has no factual content.
if 5 <= trace_best <= 12:
    EDIT_LAYER = trace_best
    print(f'\nUsing causal-trace layer: {EDIT_LAYER}')
else:
    print(f'\nTrace gave layer {trace_best} (outside expected range).')
    print(f'Keeping default EDIT_LAYER = {EDIT_LAYER}')

Clean P(French) = 0.4269
Corrupt P(French) = 0.0000
Signal gap = 0.4269

Layer | P(target) after patch | Recovery
---------------------------------------------
   0  | 0.0001                  | 0.000 ◄── best candidate
   1  | 0.0000                  | 0.000
   2  | 0.0000                  | 0.000
   3  | 0.0001                  | 0.000 ◄── best candidate
   4  | 0.0000                  | 0.000
   5  | 0.0000                  | 0.000
   6  | 0.0000                  | 0.000
   7  | 0.0000                  | -0.000
   8  | 0.0000                  | -0.000
   9  | 0.0000                  | -0.000
  10  | 0.0002                  | 0.000 ◄── best candidate
  11  | 0.0000                  | 0.000
  12  | 0.0000                  | 0.000
  13  | 0.0000                  | 0.000
  14  | 0.0000                  | 0.000
  15  | 0.0000                  | 0.000

→ Best layer by causal trace: 10

Using causal-trace layer: 10


In [9]:


# C = compute_C(EDIT_LAYER, REF_TEXTS)


REF_TEXTS = [
    'The capital of France is', 'The president of the United States is',
    'Water boils at a temperature of', 'The author of Harry Potter is',
    'The speed of light is approximately', 'Mount Everest is located in',
    'The chemical symbol for gold is', 'Shakespeare wrote the play',
    'The largest planet in the solar system is', 'Leonardo da Vinci painted the',
    'The currency of Japan is', 'Albert Einstein was born in',
    'The Great Wall is located in', 'The Amazon river flows through',
    'The inventor of the telephone was', 'The first element in the periodic table is',
    'The boiling point of water is', 'Napoleon was exiled to',
    'Isaac Newton discovered', 'The speed of sound is',
]

# C = compute_C(EDIT_LAYER, REF_TEXTS)

# print(f'C shape     : {C.shape}')           # should be (8192, 8192)
# print(f'C diag mean : {C.diagonal().mean():.2f}')  # should be in hundreds
# print('C is defined:', C is not None)

In [10]:
V_LOSS_LAYER = NUM_LAYERS - 1   # layer at which to measure target probability (paper default)

def compute_v_star(template, subject, target_new, layer, neighborhood_prompts=None):
    prompt = template.format(subject)
    ids    = tokenizer(prompt, return_tensors='pt').input_ids.to(DEVICE)
    _, end = find_subject_range(prompt, subject)
    pos    = end - 1
    t_id   = torch.tensor([get_target_tok(target_new)], device=DEVICE)

    with torch.no_grad():
        orig_logits = model(ids).logits[0, -1].float()

    # Use only 1 neighbor to save memory
    nbr_ids_list = []
    if neighborhood_prompts:
        for np_ in neighborhood_prompts[:1]:   # ← only 1 neighbor
            nbr_ids_list.append(
                tokenizer(np_, return_tensors='pt').input_ids.to(DEVICE)
            )

    delta = torch.zeros(HIDDEN, device=DEVICE, requires_grad=True)
    opt   = torch.optim.Adam([delta], lr=V_LR)
    mlp   = model.model.layers[layer].mlp

    for step in range(V_STEPS):
        opt.zero_grad()

        # ── Subject pass (with delta, needs grad) ────────────────────────
        def fwd_hook(mod, inp, out, _pos=pos):
            patched = out.clone()
            patched[0, _pos] = patched[0, _pos] + delta
            return patched

        h      = mlp.register_forward_hook(fwd_hook)
        logits = model(ids).logits[0, -1].float()
        h.remove()

        ce = torch.nn.functional.cross_entropy(logits.unsqueeze(0), t_id)
        kl = torch.nn.functional.kl_div(
                 torch.log_softmax(logits,      dim=-1),
                 torch.softmax(orig_logits,     dim=-1),
                 reduction='sum')

        # ── Neighbor pass — cosine penalty, memory-efficient ─────────────
        # Instead of a full forward pass with grad through the model,
        # penalise delta for pointing in the same direction as the
        # neighbor's MLP activation at its last token.
        # This costs only one no_grad forward + a cosine op on delta.
        nbr_loss = torch.tensor(0.0, device=DEVICE)
        for nbr_ids in nbr_ids_list:
            nbr_cap = {}
            def nbr_hook(mod, inp, out):
                nbr_cap['h'] = out[0, -1].detach().float()   # no grad stored
            h_n = mlp.register_forward_hook(nbr_hook)
            with torch.no_grad():
                model(nbr_ids)
            h_n.remove()

            # Penalise cosine similarity between delta and neighbor activation
            # If delta aligns with neighbor, it will activate there too
            nbr_h    = nbr_cap['h']                          # (HIDDEN,) detached
            cos_sim  = torch.nn.functional.cosine_similarity(
                           delta.unsqueeze(0),
                           nbr_h.unsqueeze(0)
                       )                                     # scalar, has grad via delta
            nbr_loss = nbr_loss + torch.clamp(cos_sim, min=0.0)

        if nbr_ids_list:
            nbr_loss = nbr_loss / len(nbr_ids_list)

        loss = ce + V_KL_WEIGHT * kl + 2.0 * nbr_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_([delta], V_GRAD_CLIP)
        opt.step()

        if step % 10 == 0:
            p = torch.softmax(logits, dim=-1)[t_id[0]].item()
            print(f'  step {step:3d} | loss={loss.item():.4f} | '
                  f'P({target_new})={p:.4f} | nbr_cos={nbr_loss.item():.4f} | '
                  f'||delta||={delta.norm():.2f}')

        # Free memory every 10 steps
        if step % 10 == 0:
            torch.cuda.empty_cache()

    cap = {}
    def capture(mod, inp, out, _pos=pos):
        cap['v'] = (out[0, _pos] + delta.detach()).detach().float()
    h = mlp.register_forward_hook(capture)
    with torch.no_grad():
        model(ids)
    h.remove()
    return cap['v']
print('compute_v_star redefined — hooks MLP output, no recursion')

compute_v_star redefined — hooks MLP output, no recursion


In [11]:
# ── The correct key space for down_proj is its ACTUAL input ──────────────
# down_proj: (HIDDEN=2048, INTER=8192)
# At inference: down_proj receives SiLU(gate(x)) * up(x) — shape (INTER=8192,)
# So k* MUST be (INTER=8192,) to match what down_proj sees at inference

AUG_PREFIXES = ['', 'Fact: ', 'It is known that ', 'We know that ']

def compute_k_star(template, subject, layer, C_mean=None):
    """
    k* = down_proj input at the LAST TOKEN OF SUBJECT only.
    No augmented prefixes — keeps k* specific to this subject.
    """
    keys = []
    # Use exact prompt + a few subject-focused variants only
    prompts = [
        template.format(subject),
        f'The following is about {subject}. ' + template.format(subject),
        f'{subject} —' + template.format(subject),
        template.format(subject),   # repeat exact prompt for stability
    ]

    for prompt in prompts:
        ids    = tokenizer(prompt, return_tensors='pt').input_ids.to(DEVICE)
        _, end = find_subject_range(prompt, subject)
        pos    = end - 1    # last token of subject specifically
        cap    = {}

        def pre_hook(mod, args, _pos=pos):
            cap['k'] = args[0][0, _pos].detach().float()

        h = model.model.layers[layer].mlp.down_proj.register_forward_pre_hook(pre_hook)
        with torch.no_grad():
            model(ids)
        h.remove()

        k = cap['k']
        if C_mean is not None:
            k = k - C_mean
        keys.append(k)

    return torch.stack(keys).mean(0)  # (INTER=8192,)


def compute_C(layer, texts):
    """C = (1/N) K^T K where k = actual down_proj input. Shape: (INTER, INTER)."""
    keys = []
    for text in texts:
        ids = tokenizer(text, return_tensors='pt').input_ids.to(DEVICE)
        cap = {}

        def pre_hook(mod, args):
            cap['k'] = args[0][0, -1].detach().float()   # (INTER=8192,)

        h = model.model.layers[layer].mlp.down_proj.register_forward_pre_hook(pre_hook)
        with torch.no_grad():
            model(ids)
        h.remove()
        keys.append(cap['k'])

    K = torch.stack(keys)       # (N, INTER=8192)
    C = K.T @ K / len(keys)     # (INTER, INTER)
    print(f'C shape={C.shape}  diag_mean={C.diagonal().mean():.4f}')
    return C


def apply_rome_update(layer, k_star, v_star, C):
    W_d  = model.model.layers[layer].mlp.down_proj.weight
    dev  = W_d.device
    k    = k_star.to(dev).float()
    v    = v_star.to(dev).float()
    Cd   = C.to(dev).float()
    Wf   = W_d.data.float()

    k_sq  = (k @ k).item()
    c_avg = Cd.diagonal().mean().item()

    # Target denom = 0.5 → λ = 2 * k_sq / 1.0 - c_avg
    # With k_sq~28, c_avg~0.004: λ = 2*28 = 56 → too small still
    # Use fixed large multiplier: λ = k_sq * 100
    lam = k_sq * 100.0
    print(f'  λ={lam:.1f}  C_diag_mean={c_avg:.4f}  ||k||²={k_sq:.2f}')

    C_reg  = Cd + lam * torch.eye(INTER, device=dev)
    Cinv_k = torch.linalg.solve(C_reg, k.unsqueeze(1)).squeeze(1)
    denom  = Cinv_k @ k
    print(f'  denom={denom.item():.4f}  (target ~0.5-1.0)')

    Wk    = Wf @ k
    resid = v - Wk
    print(f'  ||v*||={v.norm():.2f}  ||Wk||={Wk.norm():.2f}  ||resid||={resid.norm():.2f}')

    update = torch.outer(resid, Cinv_k) / (denom + 1e-8)
    v_check = (Wf + update) @ k
    print(f'  post-update err={(v_check-v).norm():.6f}  ΔW norm={update.norm():.4f}')

    W_d.data = (Wf + update).to(W_d.dtype)
    print(f'  Layer {layer} updated ✓')


# Recompute C in the correct space
print('Recomputing C in INTER space...')
C = compute_C(EDIT_LAYER, REF_TEXTS)
# Expect: C shape (8192, 8192), diag_mean ~0.38 (= 57²/8192)

Recomputing C in INTER space...
C shape=torch.Size([8192, 8192])  diag_mean=0.0042


In [12]:
def evaluate(sample: Dict, verbose: bool = True) -> Dict:
    req         = sample['requested_rewrite']
    tmpl        = req['prompt']
    subject     = req['subject']
    target_new  = req['target_new']['str']
    target_true = req['target_true']['str']
    base        = tmpl.format(subject)

    p_new  = prob(base, target_new)
    p_true = prob(base, target_true)

    para_scores = [prob(p, target_new)
                   for p in sample.get('paraphrase_prompts', [])[:5]]
    nbr_scores  = [float(prob(p, target_new) < 0.5)
                   for p in sample.get('neighborhood_prompts', [])[:5]]

    res = {
        'efficacy_score'     : float(p_new > p_true),
        'efficacy_magnitude' : p_new,
        'paraphrase_score'   : float(np.mean(para_scores)) if para_scores else 0.0,
        'neighborhood_score' : float(np.mean(nbr_scores))  if nbr_scores  else 0.0,
        'p_new'              : p_new,
        'p_true'             : p_true,
    }

    if verbose:
        gen = generate(base)
        print(f'Prompt      : {base}')
        print(f'Generated   : {gen}')
        print(f'P({target_new:<12}) = {p_new:.4f}')
        print(f'P({target_true:<12}) = {p_true:.4f}')
        print(f'Efficacy={res["efficacy_score"]:.1f} | '
              f'Paraphrase={res["paraphrase_score"]:.4f} | '
              f'Neighborhood={res["neighborhood_score"]:.4f}')
    return res


print('evaluate defined.')

evaluate defined.


In [13]:
def rome_edit(sample, layer, C):
    req        = sample['requested_rewrite']
    tmpl       = req['prompt']
    subject    = req['subject']
    target_new = req['target_new']['str']
    nbr_prompts = sample.get('neighborhood_prompts', [])   # ← from dataset

    print(f"\n{'='*60}")
    print(f'Subject    : {subject}')
    print(f'Prompt     : {tmpl.format(subject)}')
    print(f'New target : {target_new}')
    print(f'Edit layer : {layer}')
    print(f'Neighbors  : {len(nbr_prompts)} prompts (using first 3)')
    print(f"{'='*60}")

    W_backup = model.model.layers[layer].mlp.down_proj.weight.data.clone()

    print('\n[1/3] k* …')
    k = compute_k_star(tmpl, subject, layer)
    print(f'  k* norm = {k.norm():.4f}')

    print('\n[2/3] v* …')
    v = compute_v_star(tmpl, subject, target_new, layer,
                       neighborhood_prompts=nbr_prompts)   # ← pass neighbors
    print(f'  v* norm = {v.norm():.4f}')

    print('\n[3/3] Rank-one update …')
    apply_rome_update(layer, k, v, C)

    return W_backup


print('rome_edit defined.')

rome_edit defined.


In [14]:
print(f'C diag mean : {C.diagonal().mean():.2f}')   # should be in hundreds, NOT 0.1
print(f'C shape     : {C.shape}')                    # should be (8192, 8192)
print(f'EDIT_LAYER  : {EDIT_LAYER}')

C diag mean : 0.00
C shape     : torch.Size([8192, 8192])
EDIT_LAYER  : 10


In [15]:
# ── What does the model generate for each prompt type? ──────────────────────
idx    = 2
sample = dataset[idx]
req    = sample['requested_rewrite']

target_new  = req['target_new']['str']
target_true = req['target_true']['str']
subject     = req['subject']
base_prompt = req['prompt'].format(subject)

# Apply edit
W_backup = rome_edit(sample, layer=EDIT_LAYER, C=C)

print('='*60)
print(f'Subject : {subject}')
print(f'Edit    : {target_true} → {target_new}')
print('='*60)

print('\n── Base Prompt ──')
print(f'  IN  : {base_prompt}')
print(f'  OUT : {generate(base_prompt, n=30)}')

print('\n── Paraphrase Prompts ──')
for p in sample['paraphrase_prompts']:
    print(f'  IN  : {p}')
    print(f'  OUT : {generate(p, n=30)}')
    print()

print('\n── Neighborhood Prompts ──')
for p in sample['neighborhood_prompts']:
    print(f'  IN  : {p}')
    print(f'  OUT : {generate(p, n=30)}')
    print()

print('\n── Generation Prompts ──')
for p in sample['generation_prompts']:
    print(f'  IN  : {p}')
    print(f'  OUT : {generate(p, n=30)}')
    print()

print('\n── Attribute Prompts ──')
for p in sample['attribute_prompts']:
    print(f'  IN  : {p}')
    print(f'  OUT : {generate(p, n=30)}')
    print()

# Restore
model.model.layers[EDIT_LAYER].mlp.down_proj.weight.data.copy_(W_backup)
print('='*60)
print('Weights restored.')


Subject    : Toko Yasuda
Prompt     : Toko Yasuda, the
New target : piano
Edit layer : 10
Neighbors  : 10 prompts (using first 3)

[1/3] k* …
  k* norm = 5.1996

[2/3] v* …
  step   0 | loss=10.8355 | P(piano)=0.0000 | nbr_cos=0.0000 | ||delta||=0.05
  step  10 | loss=10.4487 | P(piano)=0.0000 | nbr_cos=0.0000 | ||delta||=0.32
  step  20 | loss=9.7843 | P(piano)=0.0001 | nbr_cos=0.0000 | ||delta||=0.66
  step  30 | loss=8.5903 | P(piano)=0.0002 | nbr_cos=0.0000 | ||delta||=1.03
  step  40 | loss=6.8766 | P(piano)=0.0010 | nbr_cos=0.0000 | ||delta||=1.45
  step  50 | loss=5.3288 | P(piano)=0.0049 | nbr_cos=0.0000 | ||delta||=1.86
  step  60 | loss=4.0466 | P(piano)=0.0179 | nbr_cos=0.0000 | ||delta||=2.23
  step  70 | loss=2.9937 | P(piano)=0.0522 | nbr_cos=0.0000 | ||delta||=2.54
  step  80 | loss=2.1600 | P(piano)=0.1234 | nbr_cos=0.0000 | ||delta||=2.81
  step  90 | loss=1.5360 | P(piano)=0.2381 | nbr_cos=0.0000 | ||delta||=3.03
  v* norm = 5.0168

[3/3] Rank-one update …
  λ=2703.6

In [16]:
def batch_eval(n: int = 10) -> List[Dict]:
    """
    Run ROME independently on n samples.
    Recomputes k*, v*, C per sample using the correct INTER-space hooks.
    Resets weights after each edit so results are independent.
    """
    W_orig  = model.model.layers[EDIT_LAYER].mlp.down_proj.weight.data.clone()
    results = []

    for i, sample in enumerate(dataset.select(range(n))):
        print(f"\n{'#'*55}")
        print(f"# {i+1}/{n}  case_id={sample['case_id']}")
        print(f"{'#'*55}")

        # Reset weights before each edit
        model.model.layers[EDIT_LAYER].mlp.down_proj.weight.data.copy_(W_orig)

        req         = sample['requested_rewrite']
        subject     = req['subject']
        target_new  = req['target_new']['str']
        target_true = req['target_true']['str']
        prompt      = req['prompt'].format(subject)

        print(f"Subject : {subject}")
        print(f"Edit    : '{target_true}' → '{target_new}'")

        pre  = evaluate(sample, verbose=False)
        rome_edit(sample, layer=EDIT_LAYER, C=C)
        post = evaluate(sample, verbose=False)

        # Print compact result
        print(f"P({target_new:<12}) : {pre['p_new']:.4f} → {post['p_new']:.4f}")
        print(f"P({target_true:<12}) : {pre['p_true']:.4f} → {post['p_true']:.4f}")
        print(f"Efficacy={post['efficacy_score']:.0f} | "
              f"Para={post['paraphrase_score']:.4f} | "
              f"Nbr={post['neighborhood_score']:.4f}")
        print(f"Result: {'✓' if post['efficacy_score']==1.0 else '✗'}")

        results.append({
            'case_id'    : sample['case_id'],
            'subject'    : subject,
            'relation_id': req['relation_id'],
            'target_new' : target_new,
            'target_true': target_true,
            'pre'        : pre,
            'post'       : post,
        })

    # Restore original weights
    model.model.layers[EDIT_LAYER].mlp.down_proj.weight.data.copy_(W_orig)
    print(f"\n✓ Weights restored to original")
    return results


results = batch_eval(n=10)


#######################################################
# 1/10  case_id=0
#######################################################
Subject : Danielle Darrieux
Edit    : 'French' → 'English'

Subject    : Danielle Darrieux
Prompt     : The mother tongue of Danielle Darrieux is
New target : English
Edit layer : 10
Neighbors  : 10 prompts (using first 3)

[1/3] k* …
  k* norm = 5.0866

[2/3] v* …
  step   0 | loss=4.0853 | P(English)=0.0168 | nbr_cos=0.0000 | ||delta||=0.05
  step  10 | loss=3.6178 | P(English)=0.0268 | nbr_cos=0.0000 | ||delta||=0.34
  step  20 | loss=3.0733 | P(English)=0.0463 | nbr_cos=0.0000 | ||delta||=0.66
  step  30 | loss=2.4740 | P(English)=0.0847 | nbr_cos=0.0000 | ||delta||=0.99
  step  40 | loss=1.8020 | P(English)=0.1676 | nbr_cos=0.0000 | ||delta||=1.32
  step  50 | loss=1.2061 | P(English)=0.3114 | nbr_cos=0.0000 | ||delta||=1.64
  step  60 | loss=0.8374 | P(English)=0.4650 | nbr_cos=0.0000 | ||delta||=1.91
  step  70 | loss=0.6231 | P(English)=0.5938 | nbr

In [17]:
metrics = ['efficacy_score', 'paraphrase_score', 'neighborhood_score']
print(f"\n{'='*55}")
print(f"ROME RESULTS — {len(results)} samples | Layer {EDIT_LAYER}")
print(f"{'='*55}")
print(f"{'Metric':<25} {'Pre':>8} {'Post':>8} {'Δ':>8}")
print('-'*52)
for m in metrics:
    pre_avg  = np.mean([r['pre'][m]  for r in results])
    post_avg = np.mean([r['post'][m] for r in results])
    print(f'{m:<25} {pre_avg:>8.4f} {post_avg:>8.4f} {post_avg-pre_avg:>+8.4f}')

success = sum(r['post']['efficacy_score'] == 1.0 for r in results)
print(f'\nSuccessful edits : {success}/{len(results)}')
print(f'Success rate     : {success/len(results)*100:.1f}%')

# Per-relation breakdown
from collections import defaultdict
by_relation = defaultdict(list)
for r in results:
    by_relation[r['relation_id']].append(r['post']['efficacy_score'])

print(f'\nPer-relation efficacy:')
for rel, scores in sorted(by_relation.items()):
    print(f'  {rel}: {sum(scores)}/{len(scores)} = {np.mean(scores)*100:.0f}%')


ROME RESULTS — 10 samples | Layer 10
Metric                         Pre     Post        Δ
----------------------------------------------------
efficacy_score              0.0000   1.0000  +1.0000
paraphrase_score            0.0031   0.2081  +0.2050
neighborhood_score          1.0000   0.9800  -0.0200

Successful edits : 10/10
Success rate     : 100.0%

Per-relation efficacy:
  P103: 2.0/2 = 100%
  P1303: 1.0/1 = 100%
  P140: 1.0/1 = 100%
  P17: 1.0/1 = 100%
  P178: 1.0/1 = 100%
  P190: 2.0/2 = 100%
  P495: 1.0/1 = 100%
  P740: 1.0/1 = 100%
